# Clustering Input Data — Norte Amazonia Bolivia

This notebook builds `output/Clustering.csv`, the input table for the k-means clustering.  
It merges geographic, economic, and infrastructure variables from several data sources into one clean table.

In [9]:
import os
import pandas as pd
import unicodedata

# Builds a normalized "DEPT_MUNICIPALITY" join key to merge tables consistently:
#   - removes accents, uppercase, replaces underscores with spaces
#   - strips " BENI" / " PANDO" suffixes that appear in some sources
def create_join_key(df):
    keys = []
    for dept, muni in zip(df["Departamento"], df["Municipio"]):
        if pd.isna(dept) or pd.isna(muni):
            keys.append("")
            continue
        dept_str = ''.join(
            c for c in unicodedata.normalize('NFD', str(dept).replace("_", " "))
            if unicodedata.category(c) != 'Mn'
        ).upper().strip()
        muni_str = ''.join(
            c for c in unicodedata.normalize('NFD', str(muni).replace("_", " "))
            if unicodedata.category(c) != 'Mn'
        ).upper().strip()
        if muni_str.endswith(" BENI"):  muni_str = muni_str[:-5].strip()
        if muni_str.endswith(" PANDO"): muni_str = muni_str[:-6].strip()
        key = f"{dept_str}_{muni_str}"
        # "Santa Rosa del Abuna" appears under two slightly different names across sources
        if key == "PANDO_SANTA ROSA DEL ABUNA":
            key = "PANDO_SANTA ROSA"
        keys.append(key)
    df["Join_Key"] = keys
    return df

## 1. Load data sources

| File | Content |
|---|---|
| `municipalities_counts.csv` | List of municipalities + household counts (reference) |
| `municipalities_database_Bolivia.csv` | Geographic, infrastructure, capacity factor data (Pablo) |
| `CSV_final.csv` | INE 2024 census data (population, vehicles, etc.) |
| `municipality_temperatures.csv` | Annual mean temperature from Renewable Ninja |

In [10]:
base_df       = pd.read_csv("../exctraction of data/output/municipalities_counts.csv", sep=",")
db_bolivia_df = pd.read_csv("data/municipalities_database_Bolivia.csv", sep=";")
csv_final_df  = pd.read_csv("../exctraction of data/output/CSV_final.csv", sep=",")
temp_df       = pd.read_csv("../renewable ninja/temperature/output/municipality_temperatures.csv", sep=",")

## 2. Normalize column names and create join keys

Each source uses slightly different column names for department and municipality. We rename everything to `Departamento` / `Municipio` and compute a normalized `Join_Key` for the merge.

In [11]:
base_df = base_df.rename(columns={"department": "Departamento", "municipality": "Municipio"})
base_df["Municipio"] = base_df["Municipio"].str.strip()
base_df = create_join_key(base_df)

db_bolivia_df = db_bolivia_df.rename(columns={"Department": "Departamento", "Municipality": "Municipio"})
db_bolivia_df["Municipio"] = db_bolivia_df["Municipio"].str.strip()
db_bolivia_df = create_join_key(db_bolivia_df)

csv_final_df = csv_final_df.rename(columns={"DEPARTAMENTO": "Departamento", "MUNICIPIO/TIOC": "Municipio"})
csv_final_df["Municipio"] = csv_final_df["Municipio"].astype(str).str.strip()
csv_final_df = create_join_key(csv_final_df)

temp_df = temp_df.rename(columns={"department": "Departamento", "municipality": "Municipio"})
temp_df["Municipio"] = temp_df["Municipio"].str.strip()
if "Departamento" not in temp_df.columns:
    dept_map = base_df.drop_duplicates("Municipio").set_index("Municipio")["Departamento"]
    temp_df["Departamento"] = temp_df["Municipio"].map(dept_map)
temp_df = create_join_key(temp_df)

## 3. Extract relevant columns from each source

| Variable | Source |
|---|---|
| `Population density` | INE, Censo Bolivia 2024: Datos y Estadísticas Clave (2025) |
| `GDP` | Andersen, Branisa, Calderón — *Estimaciones del PIB per cápita...*, CIS La Paz (2019) ⚠️ *source à confirmer* |
| `Cp_PV` | Pfenninger & Staffell, *Energy* 114 (2016) 1251–1265 — Renewables.ninja |
| `Cp_Wind` | Staffell & Pfenninger, *Energy* 114 (2016) 1224–1239 — Renewables.ninja |
| `Temp` | Renewables.ninja, données horaires 2019 (Valentine — remplace valeur Pablo) |
| `Altitude` | Bhushan et al., *ISPRS J. Photogramm.* 173 (2021) 151–165 — Planet SkySat DEM |
| `Relative_humidity` | ERA5, Copernicus CDS — données horaires 1940–présent (extrait nov. 2024) |
| `EL_network_access` | AETN, Anuario Estadístico 2024 — présence physique réseau SIN (≠ taux de raccordement des ménages) |
| `FG_network_access` | ANH, *Sistema de transporte por gasoductos* (2023) — corrections manuelles en §4b |
| `Num_vehicles` | INE, Censo Bolivia 2024 — (ménages avec auto + ménages avec moto) / population totale |
| `EL_demand` | Andersen et al. (2019) via base Pablo — consommation ENDE mesurée / population |

In [12]:
COL_VEH_AUTO = "NÚMERO DE HOGARES CON VEHÍCULOS | 2024 | Vehículo automotor | Tiene"
COL_VEH_MOTO = "NÚMERO DE HOGARES CON VEHÍCULOS | 2024 | Motocicleta o cuadratrack | Tiene"
COL_TOT_POP  = "NÚMERO DE PERSONAS POR FUENTE DE ELECTRICIDAD | 2024 | Total"

# Les lignes agrégées (Amazonia Norte total) ont Municipio = "nan" après le astype(str) ci-dessus
csv_muni_df = csv_final_df[csv_final_df["Municipio"] != "nan"]

vehicles_sub = csv_muni_df[["Join_Key", COL_VEH_AUTO, COL_VEH_MOTO, COL_TOT_POP]].copy()
vehicles_sub["Num_vehicles"] = (
    (vehicles_sub[COL_VEH_AUTO] + vehicles_sub[COL_VEH_MOTO])
    / vehicles_sub[COL_TOT_POP]
).round(4)
vehicles_sub = vehicles_sub[["Join_Key", "Num_vehicles"]]

temp_sub = temp_df[["Join_Key", "annual_mean_temp"]].rename(columns={"annual_mean_temp": "Temp"})

db_sub = db_bolivia_df[[
    "Join_Key", "Latitude", "Longitude", "Population density",
    "GDP", "Cp_PV", "Cp_Wind", "Altitude", "Relative_humidity",
    "EL_network_access", "FG_network_access", "EL_demand"
]].copy()

## 4. Merge all sources

Left join on `Join_Key` to keep all 21 municipalities from `base_df`, even if some data is missing.

In [13]:
df = base_df.merge(db_sub,      on="Join_Key", how="left")
df = df.merge(vehicles_sub,     on="Join_Key", how="left")
df = df.merge(temp_sub,         on="Join_Key", how="left")

# Remove the summary "TOTAL" row and duplicates from the merge
df = df[df["Departamento"] != "Amazonia_Norte"]
df = df.drop_duplicates(subset=["Departamento", "Municipio"], keep="first")

## 4b. Corrections manuelles des flags d'infrastructure

Après le merge, certains flags sont absents ou incorrects : `Exaltación` n'est pas dans la base Pablo, et `FG_network_access` ne reflète pas la carte des gasoductos ANH (2023).

| Municipalité | Dept | EL_network_access | FG_network_access | Raison |
|---|---|:---:|:---:|---|
| Exaltación | Beni | 1 | — | connecté au SIN |
| San_Lorenzo | Pando | — | 0 | FG=0 (incorrect) |
| Cobija | Pando | — | 1 | Réseau de distribution gaz confirmé par carte ANH |
| Riberalta | Beni | — | 1 | Réseau de distribution gaz confirmé par carte ANH |
| Guayaramerín | Beni | — | 1 | Réseau de distribution gaz confirmé par carte ANH |

**Source FG** : ANH, *Sistema de transporte por gasoductos* (2023) — [PDF](https://www.anh.gob.bo/InsideFiles/Documentos/Documentos_Id-1037-260130-0307-0.pdf)

<div style="background:#fff3cd; border-left:5px solid #ffc107; padding:10px; border-radius:4px; margin:8px 0">
⚠️ <b>À vérifier</b> : San_Lorenzo (Pando) <code>FG_network_access = 1</code> — à confirmer sur le terrain.
</div>

In [14]:
corrections = {
    ("Beni",  "Exaltación"):   {"EL_network_access": 1.0},
    ("Pando", "San_Lorenzo"):  {"FG_network_access": 0.0},
    ("Pando", "Cobija"):       {"FG_network_access": 1.0},
    ("Beni",  "Riberalta"):    {"FG_network_access": 1.0},
    ("Beni",  "Guayaramerín"): {"FG_network_access": 1.0},
}
for (dept, muni), flags in corrections.items():
    mask = (df["Departamento"] == dept) & (df["Municipio"] == muni)
    for col, val in flags.items():
        df.loc[mask, col] = val

## 5. EL_demand — source and known limitations

`EL_demand` (MWh/person/year) comes **directly from Pablo's database** (ENDE metered consumption / total population).  
The RAMP-based recalculation that previously added sufficiency demand for non-electrified households is **commented out** below; it is not needed for the clustering step, which should reflect current observed demand patterns.

**Known limitation**: for 5 municipalities where ENDE has no metered records (Bolpebra, Ingavi, Santos_Mercado, Exaltación, Santa_Rosa_Pando), `EL_demand = 0`. Off-grid consumption (solar panels, private generators) is not captured by ENDE statistics.

<div style="background:#d1ecf1; border-left:5px solid #17a2b8; padding:10px; border-radius:4px; margin:8px 0">
ℹ️ <b>Source</b>: Andersen, Branisa, Calderón (2019), <em>Estimaciones del PIB per cápita y de la actividad económica a nivel municipal en Bolivia en base a datos de consumo de electricidad</em>, La Paz: CIS.
</div>

<div style="background:#fff3cd; border-left:5px solid #ffc107; padding:10px; border-radius:4px; margin:8px 0">
⚠️ <b>Attention</b>: The <code>EL_demand</code> column in <code>municipalities_database_Bolivia.csv</code> doesn't represent the <em>total</em> electricity demand of the municipality!
</div>

In [15]:
# Recalcul RAMP commenté — EL_demand vient directement de la base Pablo (voir §5 ci-dessus)

# RAMP_ELEC_COLS = [
#     "sufficiency_illumination", "sufficiency_ICT", "sufficiency_cold_storage",
#     "sufficiency_thermal_comfort",
#     "big_school_illumination", "big_school_ICT", "big_school_cold_storage", "big_school_space_cooling",
#     "health_center_illumination", "health_center_ICT", "health_center_cold_storage",
#     "health_center_space_cooling", "health_center_water_heating",
#     "health_center_water_supply", "health_center_medical_equip",
#     "entertainment_business_illumination", "entertainment_business_ICT", "entertainment_business_cold_storage",
#     "restaurant_illumination", "restaurant_cold_storage", "restaurant_kitchen",
#     "store_illumination", "store_ICT", "store_cold_storage",
#     "workshop_illumination", "workshop_ICT", "workshop_machinery",
#     "public_lighting_illumination",
#     "rice_processing_rice_processing",
# ]
#
# el_pablo = df.set_index("Join_Key")["EL_demand"].fillna(0.0)
#
# COL_TOT_POP = "NÚMERO DE PERSONAS POR FUENTE DE ELECTRICIDAD | 2024 | Total"
# df = df.merge(
#     csv_final_df[["Join_Key", COL_TOT_POP]].rename(columns={COL_TOT_POP: "_total_pop"}),
#     on="Join_Key", how="left"
# )
#
# def compute_ramp_total_mwh(muni_name):
#     path = f"../analyse data ramp/data ramp/{muni_name}/load_curve_energy_service_full_year_Norte_Amazonia.csv"
#     if not os.path.exists(path):
#         return None
#     df_ramp = pd.read_csv(path)
#     cols = [c for c in RAMP_ELEC_COLS if c in df_ramp.columns]
#     return df_ramp[cols].fillna(0).values.sum() / 60 / 1_000_000
#
# for idx, row in df.iterrows():
#     muni         = row["Municipio"]
#     el_pablo_val = el_pablo.get(row["Join_Key"], 0.0)
#     tot_pop      = row["_total_pop"]
#     ramp_mwh     = compute_ramp_total_mwh(muni)
#
#     if ramp_mwh is None:
#         print(f"Warning: no RAMP data for '{muni}' — keeping original EL_demand value.")
#     elif pd.isna(tot_pop) or tot_pop == 0:
#         print(f"Warning: missing total population for '{muni}' — keeping original EL_demand value.")
#     else:
#         df.at[idx, "EL_demand"] = round(el_pablo_val + (ramp_mwh / tot_pop), 3)

## 6. Export

Final column order and save to `output/Clustering.csv`.

In [16]:
FINAL_COLUMNS = [
    "Departamento", "Municipio", "Latitude", "Longitude",
    "Population density", "GDP", "Cp_PV", "Cp_Wind", "Temp",
    "Altitude", "Relative_humidity", "Num_vehicles",
    "EL_network_access", "FG_network_access", "EL_demand",
]

df[FINAL_COLUMNS].to_csv("output/Clustering.csv", index=False, sep=";")
print("Done! Clustering.csv saved.")

Done! Clustering.csv saved.
